# Cross-Country Solar Farm Comparison

This notebook compares solar farm data across Benin, Sierra Leone, and Togo to identify relative solar potential and key differences.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

## 1. Load Cleaned Data from All Countries

In [ ]:
benin = pd.read_csv('../data/benin_clean.csv')
benin['Country'] = 'Benin'
print(f"Benin shape: {benin.shape}")

sierraleone = pd.read_csv('../data/sierraleone_clean.csv')
sierraleone['Country'] = 'Sierra Leone'
print(f"Sierra Leone shape: {sierraleone.shape}")

togo = pd.read_csv('../data/togo_clean.csv')
togo['Country'] = 'Togo'
print(f"Togo shape: {togo.shape}")

all_data = pd.concat([benin, sierraleone, togo], ignore_index=True)
print(f"\nCombined dataset shape: {all_data.shape}")

## 2. Summary Statistics Comparison

In [ ]:
summary_metrics = ['GHI', 'DNI', 'DHI']
summary_table = all_data.groupby('Country')[summary_metrics].agg(['mean', 'median', 'std'])
print("Summary Statistics by Country:")
print(summary_table)

summary_table.to_csv('../data/country_comparison_summary.csv')
print("\nSummary table saved to ../data/country_comparison_summary.csv")

## 3. Boxplot Comparison - GHI, DNI, DHI

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

sns.boxplot(data=all_data, x='Country', y='GHI', ax=axes[0])
axes[0].set_title('GHI Comparison Across Countries', fontsize=14, fontweight='bold')
axes[0].set_ylabel('GHI (W/m²)')
axes[0].set_xlabel('Country')

sns.boxplot(data=all_data, x='Country', y='DNI', ax=axes[1])
axes[1].set_title('DNI Comparison Across Countries', fontsize=14, fontweight='bold')
axes[1].set_ylabel('DNI (W/m²)')
axes[1].set_xlabel('Country')

sns.boxplot(data=all_data, x='Country', y='DHI', ax=axes[2])
axes[2].set_title('DHI Comparison Across Countries', fontsize=14, fontweight='bold')
axes[2].set_ylabel('DHI (W/m²)')
axes[2].set_xlabel('Country')

plt.tight_layout()
plt.show()

## 4. Statistical Testing - ANOVA

In [ ]:
print("One-Way ANOVA Test Results:")
print("="*60)

for metric in ['GHI', 'DNI', 'DHI']:
    benin_data = benin[metric].dropna()
    sl_data = sierraleone[metric].dropna()
    togo_data = togo[metric].dropna()
    
    f_stat, p_value = stats.f_oneway(benin_data, sl_data, togo_data)
    
    print(f"\n{metric}:")
    print(f"  F-statistic: {f_stat:.4f}")
    print(f"  p-value: {p_value:.6f}")
    
    if p_value < 0.05:
        print(f"  Result: Significant difference (p < 0.05)")
    else:
        print(f"  Result: No significant difference (p >= 0.05)")

print("\n" + "="*60)

## 5. Bar Chart - Average GHI Ranking

In [ ]:
avg_ghi = all_data.groupby('Country')['GHI'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
avg_ghi.plot(kind='bar', ax=ax, color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
ax.set_title('Average GHI by Country (Ranked)', fontsize=16, fontweight='bold')
ax.set_ylabel('Average GHI (W/m²)', fontsize=12)
ax.set_xlabel('Country', fontsize=12)
ax.set_xticklabels(avg_ghi.index, rotation=0)

for i, v in enumerate(avg_ghi.values):
    ax.text(i, v + 5, f'{v:.2f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print("\nAverage GHI Ranking:")
for i, (country, value) in enumerate(avg_ghi.items(), 1):
    print(f"{i}. {country}: {value:.2f} W/m²")

## 6. Multi-Metric Comparison

In [ ]:
avg_metrics = all_data.groupby('Country')[['GHI', 'DNI', 'DHI']].mean()

fig, ax = plt.subplots(figsize=(12, 6))
avg_metrics.plot(kind='bar', ax=ax)
ax.set_title('Average Solar Irradiance Metrics by Country', fontsize=16, fontweight='bold')
ax.set_ylabel('Irradiance (W/m²)', fontsize=12)
ax.set_xlabel('Country', fontsize=12)
ax.set_xticklabels(avg_metrics.index, rotation=0)
ax.legend(['GHI', 'DNI', 'DHI'], loc='upper right')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Temperature Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.boxplot(data=all_data, x='Country', y='Tamb', ax=axes[0])
axes[0].set_title('Ambient Temperature Comparison', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Temperature (°C)')
axes[0].set_xlabel('Country')

avg_temp = all_data.groupby('Country')['Tamb'].mean().sort_values(ascending=False)
avg_temp.plot(kind='bar', ax=axes[1], color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
axes[1].set_title('Average Temperature by Country', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Average Temperature (°C)')
axes[1].set_xlabel('Country')
axes[1].set_xticklabels(avg_temp.index, rotation=0)

plt.tight_layout()
plt.show()

## 8. Humidity and Wind Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.boxplot(data=all_data, x='Country', y='RH', ax=axes[0])
axes[0].set_title('Relative Humidity Comparison', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Relative Humidity (%)')
axes[0].set_xlabel('Country')

sns.boxplot(data=all_data, x='Country', y='WS', ax=axes[1])
axes[1].set_title('Wind Speed Comparison', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Wind Speed (m/s)')
axes[1].set_xlabel('Country')

plt.tight_layout()
plt.show()

## 9. Correlation Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

corr_cols = ['GHI', 'DNI', 'DHI', 'Tamb', 'RH']
available_cols = [col for col in corr_cols if col in benin.columns]

sns.heatmap(benin[available_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, ax=axes[0], cbar_kws={'shrink': 0.8})
axes[0].set_title('Benin - Correlation Matrix', fontweight='bold')

sns.heatmap(sierraleone[available_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, ax=axes[1], cbar_kws={'shrink': 0.8})
axes[1].set_title('Sierra Leone - Correlation Matrix', fontweight='bold')

sns.heatmap(togo[available_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, ax=axes[2], cbar_kws={'shrink': 0.8})
axes[2].set_title('Togo - Correlation Matrix', fontweight='bold')

plt.tight_layout()
plt.show()

## 10. Distribution Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for country, color in zip(['Benin', 'Sierra Leone', 'Togo'], ['blue', 'green', 'red']):
    country_data = all_data[all_data['Country'] == country]
    axes[0].hist(country_data['GHI'], bins=50, alpha=0.5, label=country, color=color)
    axes[1].hist(country_data['DNI'], bins=50, alpha=0.5, label=country, color=color)
    axes[2].hist(country_data['DHI'], bins=50, alpha=0.5, label=country, color=color)

axes[0].set_title('GHI Distribution', fontweight='bold')
axes[0].set_xlabel('GHI (W/m²)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

axes[1].set_title('DNI Distribution', fontweight='bold')
axes[1].set_xlabel('DNI (W/m²)')
axes[1].set_ylabel('Frequency')
axes[1].legend()

axes[2].set_title('DHI Distribution', fontweight='bold')
axes[2].set_xlabel('DHI (W/m²)')
axes[2].set_ylabel('Frequency')
axes[2].legend()

plt.tight_layout()
plt.show()

## Key Observations

### Solar Radiation Potential:
1. **Highest GHI**: [Country with highest average GHI]
2. **Most Consistent**: [Country with lowest standard deviation]
3. **Best DNI**: [Country with highest DNI for concentrated solar]

### Environmental Factors:
1. **Temperature**: [Observations about temperature differences]
2. **Humidity**: [Impact of humidity on solar potential]
3. **Wind Patterns**: [Wind speed variations across countries]

### Recommendations:
1. **Primary Investment**: [Best country for solar installation based on data]
2. **Technology Fit**: [Which solar technology suits each region]
3. **Risk Assessment**: [Variability and reliability considerations]